In [1]:


# Cell 1: Setup path and imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src import CustomBPETokenizer, normalize_review
from datasets import load_dataset
from src.data import create_dataloaders
from src.model import TextEmbeddings

# Cell 2+: Use classes directly
my_tokenizer = CustomBPETokenizer()
model_path = Path.cwd().parent / "data" / "models" / "tokenizer_state.json"


c:\Users\pc\anaconda3\envs\drishti\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_dataset = load_dataset("stanfordnlp/imdb")

In [3]:

train_val_split = raw_dataset["train"].train_test_split(test_size=0.2, seed=42)

datasets = {
    "train": train_val_split["train"],
    "validation": train_val_split["test"],
    "test": raw_dataset["test"]
}

print(f"Train size: {len(datasets['train'])}")
print(f"Validation size: {len(datasets['validation'])}")
print(f"Test size: {len(datasets['test'])}\n")



Train size: 20000
Validation size: 5000
Test size: 25000



In [4]:
cleaned_datasets = {}
for split_name, split_data in datasets.items():
    cleaned_datasets[split_name] = split_data.map(normalize_review, num_proc=4)

print("\n--- Normalization Complete ---")
cleaned_datasets


--- Normalization Complete ---


{'train': Dataset({
     features: ['text', 'label'],
     num_rows: 20000
 }),
 'validation': Dataset({
     features: ['text', 'label'],
     num_rows: 5000
 }),
 'test': Dataset({
     features: ['text', 'label'],
     num_rows: 25000
 })}

In [5]:
corpus = []
corpus = [i for i in cleaned_datasets['train']['text']]


In [6]:

my_tokenizer.load_or_train(corpus=corpus, vocab_size=10000, file_path=str(model_path))

Tokenizer state loaded from c:\Users\pc\Desktop\LLMs\data\models\tokenizer_state.json. Vocabulary size: 10034


In [7]:

test_text = "This is a good Movie, I like Avengers, but I don't like the ending. The acting was great, but the plot was a bit weak."

input_ids = my_tokenizer.encode(test_text)
print("Input IDs:", input_ids)
print("Number of tokens:", len(input_ids))
reconstructed_text = my_tokenizer.decode(input_ids)
print("Reconstructed:", reconstructed_text)

Input IDs: [2, 791, 147, 109, 328, 206, 42, 116, 275, 8118, 4362, 42, 207, 116, 435, 47, 5, 275, 113, 1042, 41, 113, 542, 184, 414, 42, 207, 113, 519, 184, 109, 751, 1779, 41, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [8]:
# 1. Extract the lists of texts and labels from your cleaned_datasets
train_texts = cleaned_datasets['train']['text']
train_labels = cleaned_datasets['train']['label']

val_texts = cleaned_datasets['validation']['text']
val_labels = cleaned_datasets['validation']['label']

test_texts = cleaned_datasets['test']['text']
test_labels = cleaned_datasets['test']['label']

train_loader, val_loader, test_loader = create_dataloaders(
    train_texts, train_labels, 
    val_texts, val_labels, 
    test_texts, test_labels, 
    tokenizer=my_tokenizer, 
    batch_size=16, 
    max_length=256  # Make sure this is 256!
)
sample_batch_tokens, sample_batch_labels = next(iter(train_loader))

# 2. Dynamically get the true vocabulary size from your tokenizer
# Note: depending on how you wrote CustomBPETokenizer, this might be 
# len(my_tokenizer.vocab) or my_tokenizer.vocab_size. Adjust if necessary!
VOCAB_SIZE = len(my_tokenizer.vocab) 
EMBEDDING_DIM = 128
MAX_LENGTH = 256 

print(f"True Vocabulary Size: {VOCAB_SIZE}")
print(f"Max ID in our batch: {sample_batch_tokens.max().item()}")

# 3. Instantiate the layer
embedding_layer = TextEmbeddings(
    vocab_size=VOCAB_SIZE, 
    embedding_dim=EMBEDDING_DIM, 
    max_length=MAX_LENGTH
)

# 4. Pass the batch through
embedded_output = embedding_layer(sample_batch_tokens)

print("Input IDs shape:", sample_batch_tokens.shape)
print("Embedded output shape:", embedded_output.shape)

True Vocabulary Size: 10034
Max ID in our batch: 9959
Input IDs shape: torch.Size([16, 256])
Embedded output shape: torch.Size([16, 256, 128])
